# Sélection maladies Orphanet

Outil d'annotation manuelle pour les patients dont le gène est associé à plusieurs maladies.

## Pipeline

```
convert_genes_to_diseases.ipynb
        →  patients_multi_diseases.csv
choice_disease.ipynb 
        →  choice_output.csv
        →  patients_one_disease.csv
```

## Utilisation

1. Générer `patients_multi_diseases.csv` avec `convert_genes_to_diseases.ipynb`.
2. Lancer la cellule **Annotation** : pour chaque patient, taper le numéro de la maladie et appuyer sur Entrée.
   - Taper `s` + Entrée pour passer un patient sans l'annoter.
   - Taper `q` + Entrée pour arrêter et sauvegarder la progression.
3. Lancer la cellule **Export final** pour produire `patients_one_disease.csv`.

In [66]:
import pandas as pd
from IPython.display import clear_output
import re
import ast


def nettoyer_html(texte):
    texte = str(texte)
    texte = re.sub(r'<[^>]+>', ' ', texte)      # supprime les balises
    texte = texte.replace('&nbsp;', ' ')         # entités HTML courantes
    texte = texte.replace('&amp;', '&')
    texte = texte.replace('&lt;', '<')
    texte = texte.replace('&gt;', '>')
    texte = re.sub(r'\s+', ' ', texte)           # multiple espaces/\n ? un seul espace
    return texte.strip()

# Dataset contenant les comptes rendus
crData = pd.read_csv("../../../data/raw_data/cr_genetique_exomes_with_text.csv", sep=";")
colonnes_cr = ["DOCUMENT_ID","PATIENT_ID","TEXT"]

INPUT_FILE  = "patients_multi_diseases.csv"
OUTPUT_FILE = "choice_output.csv"

df = pd.read_csv(INPUT_FILE)
all_ids = list(df["ID_PAT_ETUDE"].unique())

try:
    done_ids = set(pd.read_csv(OUTPUT_FILE)["ID_PAT_ETUDE"].unique())
except FileNotFoundError:
    done_ids = set()

all_ids = sorted(list(df["ID_PAT_ETUDE"].unique()))
remaining = [p for p in all_ids if p not in done_ids]

print("Total patients :", len(all_ids))
print("Deja annotes   :", len(done_ids))
print("Restants       :", len(remaining))
if done_ids:
    print("Reprise au patient", len(done_ids) + 1)

Total patients : 232
Deja annotes   : 0
Restants       : 232


In [61]:
print(len(remaining))
print(remaining[:232])

232
['002531B316C38737CCB3E253033752FF2F5386B1', '00918078EC35C2F65A4C7E0A3F0F4C82E17EF661', '00B41C730BE1F8A8EC7FE72C2E136756E7661DDF', '01CB157ADB17F3F4AD2982D71ED41BEADA185229', '08D54A33A83900DE6C0AF08D82AB445D6090565E', '08EC915DCAAB79B1A282BC0F00350848E8C2A256', '0AC05B34B7C7EC7942BDCB818BAAED79F979FE52', '0B2336C8ECE6539245958485E7BC03E4C26B2F0B', '0B73C3C38761C435B7F604A5EDFF604CD4359531', '0D1BB26F1405B8F834CB54319007CC4AA2B59D4A', '0E701156B5DBA1861D035D81EAF941E34857A2BA', '0EC0D5C1FD324D7C88B07ECFFB4029F13A2F41F8', '0EE11358EBCC09E284E9F968EA6DB62E42929D39', '0F1E15F1331F803A28C1DDFF8598195DAE341562', '117ADB48CB801984163164517A3BBA5AA6271E53', '11FD087E3E908F0D6E8E2B8823036BE6446064D2', '1463CDBDE8BEE42C4111A91569ED45547DD759CB', '14AB4B090CD29D3B45293C1450551122D48AE0C3', '14B0073E4BDF36C06B151600A9C1A95053909386', '15F4CA5B976260C4E527AFF8A031A36322FA25B4', '1670733021C2944396D83DEF99F95B65400C8DAD', '16891613456925F7A3A0F5E602EBCD6B0DAA5E0B', '17618FCB122E6E794081E26808

## Annotation

Lancez la cellule ci-dessous. Pour chaque patient :
- tapez le **numéro** de la maladie choisie et appuyez sur **Entrée**
- tapez **`s`** pour passer le patient
- tapez **`q`** pour arrêter (la progression est sauvegardée)

In [67]:
def save_result(pat_id, disease_name, orpha_code):
    new_row = pd.DataFrame([{
        "ID_PAT_ETUDE"     : pat_id,
        "real_disease_name": disease_name,
        "real_disease_code": orpha_code,
    }])
    try:
        current = pd.read_csv(OUTPUT_FILE)
        updated = pd.concat([current, new_row]).drop_duplicates("ID_PAT_ETUDE", keep="last")
    except FileNotFoundError:
        updated = new_row
    updated.to_csv(OUTPUT_FILE, index=False)


SEP = "-" * 60

remaining = sorted(remaining, reverse=False)

for i, pat_id in enumerate(remaining):
    pat_data = df[df["ID_PAT_ETUDE"] == pat_id]

    gene    = ", ".join(pat_data["gene_symbol"].unique())
    ipp     = pat_data["IPP_clef"].iloc[0]
    gene_id = ", ".join(pat_data["GeneId"].unique().astype(str))
    hpo     = (
        pat_data["hpo_unique_list_name"].iloc[0]
        if "hpo_unique_list_name" in pat_data.columns and pd.notna(pat_data["hpo_unique_list_name"].iloc[0])
        else "—"
    )

    choices = pat_data[["OrphaCode", "DiseaseName"]].drop_duplicates().reset_index(drop=True)

    # Affichage des comptes rendus
    crs_patient = crData[crData["PATIENT_ID"] == pat_id][["DOCUMENT_ID", "TEXT"]].reset_index(drop=True)
    textes_cr   = [nettoyer_html(t) for t in crs_patient["TEXT"].tolist()]
    doc_ids     = crs_patient["DOCUMENT_ID"].tolist()

    cr_i = 0
    while True:
        clear_output(wait=True)
        print(SEP)
        print("  Patient", i + 1, "/", len(remaining))
        print(SEP)
        print("  Patient ID :", pat_id)
        print("  IPP_clef   :", ipp)
        print("  Gene       :", gene)
        print("  Gene ID    :", gene_id)
        print("  Termes HPO :", hpo)
        print()
        if textes_cr:
            print(f"  --- Compte rendu {cr_i + 1} / {len(textes_cr)} --- Document ID : {doc_ids[cr_i]}")
            print(textes_cr[cr_i])
        else:
            print("  (aucun compte rendu)")
        print()
        print("  Maladies proposees :")
        for j, row in choices.iterrows():
            print("   ", j+1, "-", row["DiseaseName"], " (ORPHA:" + str(row["OrphaCode"]) + ")")
        print()
        print("  [n] CR suivant   [p] CR precedent   [s] Passer   [q] Quitter   ou un numero")
        print(SEP)

        rep = input("  Votre choix : ").strip().lower()

        if rep == "n" and textes_cr and cr_i < len(textes_cr) - 1:
            cr_i += 1
        elif rep == "p" and cr_i > 0:
            cr_i -= 1
        elif rep == "q":
            print("Annotation arretee. Relancez pour reprendre.")
            break
        elif rep == "s":
            print("  -> Patient", pat_id, "passe.")
            break
        elif rep.isdigit() and 1 <= int(rep) <= len(choices):
            idx_choice = int(rep) - 1
            row = choices.iloc[idx_choice]
            save_result(pat_id, row["DiseaseName"], row["OrphaCode"])
            print("  OK Sauvegarde :", row["DiseaseName"])
            break

    if rep == "q":
        break

else:
    clear_output(wait=True)
    print("Tous les patients ont ete annotes !")
    print("Fichier :", OUTPUT_FILE)


------------------------------------------------------------
  Patient 232 / 232
------------------------------------------------------------
  Patient ID: 002531B316C38737CCB3E253033752FF2F5386B1
  IPP_clef: 18655269
  Gene symbole   : NIPBL
  Gene ID :  16548
  Termes HPO: ['anomaly of communication', 'synophris', 'single milk coffee task at the abdominal level', 'autism', 'enterocolitis', 'behavioral disorders', 'structure abnormalities of the kidneys', 'pectus excavatum', 'look at the nose', 'behavioural and social impairments', 'delay in language', 'late puberty', 'kidney structural abnormalities', 'mania', 'balance difficulties', 'Cardiac malformations', 'anxiety', 'relative hyperglycaemia', 'autistic spectrum disorder', 'autism spectrum disorders', 'significant frustration intolerance', 'bicorne uterus', 'actually significant hairiness', 'repetitive behaviors', 'speech and severe eating difficulties', 'significant difficulties in fine motor skills', 'intellectual disability?', '

  Votre choix :  2


DEBUG rep = '2'


In [46]:
pat_test = remaining[0]
print("Premier patient :", pat_test)
print("Nombre de CR :", len(crData[crData["PATIENT_ID"] == pat_test]))
print("IDs documents :", crData[crData["PATIENT_ID"] == pat_test]["DOCUMENT_ID"].tolist())


Premier patient : 002531B316C38737CCB3E253033752FF2F5386B1
Nombre de CR : 4
IDs documents : ['F814031695A370478EF2B1EECD6B4E95C67C56AB', 'B3A76A229F8194EBDD2035006EADE147B6912779', '8C067ABF77C823773A685DE65A83240E7DDFE9E6', '14E27E21DEF7393B90B5E4185065E3CCB4C19F83']


## Export final

Relancez la cellule ci-dessous pour joindre les nouvelles annotations au fichier d'entrée.

In [ ]:
annotations = pd.read_csv(OUTPUT_FILE)

drop_cols = [c for c in ["OrphaCode", "DiseaseName", "GeneSymbol", "GeneId"] if c in df.columns]
df_patients = df.drop_duplicates("ID_PAT_ETUDE")[[c for c in df.columns if c not in drop_cols]]

df_export = df_patients.merge(annotations, on="ID_PAT_ETUDE", how="left")
df_export.to_csv("patients_one_disease.csv", index=False)

annotated = df_export["real_disease_code"].notna().sum()
print("Export : patients_one_disease.csv")
print("Annotes :", annotated, "/", len(df_export))
df_export.head()